In [ ]:
import jax
import jax.numpy as jnp
import sys  
import numpy as np
import matplotlib.pyplot as plt 
import jax.config as jax_config
import os
path = "../TRAINING"
os.chdir(path)
os.listdir(path)
from model import create_model
jax_config.update("jax_enable_x64", True)

In [ ]:
DIM = 2
TRAIN_ITERATION = 10000
BATCH_SIZE = 100
DT = 1e-2
FEATURE_NET_LAYER = [12,24, 48]
DESCRIBE_NET_LAYER = [128, 128, 128, DIM]

params = jnp.load(f"./model_save_path/jax_ckpt_000100.npz", allow_pickle=True)['params'].item()
_, apply_fn = create_model(FEATURE_NET_LAYER, DESCRIBE_NET_LAYER, DIM)

In [ ]:
def ic_density_step(N, d=2, box=2.0, bias=0.7, seed=0):
    # bias 越大，左半区越密
    rng=np.random.default_rng(seed)
    nL=int(N*bias); nR=N-nL
    xL=np.c_[rng.uniform(-box,0,(nL,1)), rng.uniform(-box,box,(nL,1))]
    xR=np.c_[rng.uniform(0, box,(nR,1)), rng.uniform(-box,box,(nR,1))]
    return np.vstack([xL,xR]).astype(np.float32)
x = jnp.array(ic_density_step(16000))

In [ ]:
plt.hist2d(x[:,0],x[:,1],bins=100);

In [ ]:
def step(xn):
    f = apply_fn(params, x)
    return xn + DT *f


In [ ]:
x_save = x[None,...]
for i in range(200):
    x = step(x)
    x_save = jnp.concatenate([x_save, x[None,...]], axis=0)


In [ ]:
plt.hist2d(x_save[0,:,0],x_save[0,:,1],bins=100);

In [ ]:
plt.hist2d(x_save[-1,:,0],x_save[-1,:,1],bins=100);